In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
data_path = 'dblp-v10.csv'
df = pd.read_csv(data_path)

target_column = 'venue'
duplicate_subset = ['id'] if 'id' in df.columns else None

print('Original shape:', df.shape)
print('Original duplicates:', int(df.duplicated(subset=duplicate_subset).sum()) if duplicate_subset else 'n/a')
print('Original missing values:', int(df.isna().sum().sum()))
print(df.head(3).to_string())

text_columns = ['abstract', 'authors', 'references', 'title', target_column]
for column in text_columns:
    df[column] = df[column].fillna('').astype(str).str.strip()

df['n_citation'] = pd.to_numeric(df['n_citation'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['n_citation'] = df['n_citation'].fillna(df['n_citation'].median()).astype(int)
df['year'] = df['year'].fillna(df['year'].median()).astype(int)

df = df.drop_duplicates(subset=duplicate_subset).copy() if duplicate_subset else df.drop_duplicates().copy()
df = df[df[target_column] != ''].copy()

for column in ['abstract', 'authors', 'references', 'title']:
    df[column] = df[column].replace({'nan': '', 'None': ''}).fillna('').astype(str).str.strip()

print('Cleaned shape:', df.shape)
print('Cleaned duplicates:', int(df.duplicated(subset=duplicate_subset).sum()) if duplicate_subset else 'n/a')
print('Remaining missing values:', int(df.isna().sum().sum()))
print('Blank venues:', int((df[target_column] == '').sum()))

Original shape: (1000000, 8)
Original duplicates: 0
Original missing values: 474641
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [6]:
# Use the most common venue classes so KNN stays practical and accurate on this large dataset.
top_n = 8
rows_per_class = 1500
venue_counts = df[target_column].value_counts()
top_venues = venue_counts.head(top_n).index
df = df[df[target_column].isin(top_venues)].copy()
df = pd.concat(
    [
        frame.sample(n=min(len(frame), rows_per_class), random_state=42)
        for _, frame in df.groupby(target_column)
    ],
    ignore_index=True,
)

def build_feature_text(frame: pd.DataFrame) -> pd.Series:
    citation_bins = pd.cut(
        frame['n_citation'],
        bins=[-1, 0, 5, 20, 100, 500, 10**9],
        labels=['zero', 'low', 'medium', 'high', 'very_high', 'extreme']
    ).astype(str)
    return (
        frame['title'].astype(str) + ' ' +
        frame['abstract'].astype(str) + ' ' +
        frame['authors'].astype(str) + ' ' +
        frame['references'].astype(str) + ' ' +
        'year_' + frame['year'].astype(str) + ' ' +
        'citation_' + citation_bins
    ).str.replace(r'\\s+', ' ', regex=True).str.strip()

# TF-IDF converts the string data into a numeric matrix that KNN can use.
df['feature_text'] = build_feature_text(df)
label_encoder = LabelEncoder()
df['venue_encoded'] = label_encoder.fit_transform(df[target_column])

print('Modeling shape:', df.shape)
print('Classes:', len(label_encoder.classes_))
print(df[target_column].value_counts().to_string())

X = df['feature_text']
y = df['venue_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

Modeling shape: (12000, 10)
Classes: 8
venue
Bioinformatics                                                          1500
Lecture Notes in Computer Science                                       1500
global communications conference                                        1500
intelligent robots and systems                                          1500
international conference on acoustics, speech, and signal processing    1500
international conference on communications                              1500
international conference on image processing                            1500
international conference on robotics and automation                     1500
Train size: 9600
Test size: 2400


In [7]:
# Training the KNN model with TF-IDF features.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('knn', KNeighborsClassifier(algorithm='brute'))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1)],
    'tfidf__max_features': [10000],
    'tfidf__min_df': [2],
    'knn__n_neighbors': [3, 5, 7],
    'knn__weights': ['distance'],
    'knn__metric': ['cosine']
}

search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1
)
search.fit(X_train, y_train)

print('Best CV accuracy:', search.best_score_)
print('Best parameters:', search.best_params_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print('Test accuracy:', test_accuracy)
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('Classification report:\n', classification_report(y_test, y_pred, target_names=label_encoder.classes_))

sample_texts = X_test.iloc[:5] if hasattr(X_test, 'iloc') else X_test[:5]
sample_predictions = label_encoder.inverse_transform(best_model.predict(sample_texts))

for index, (text, predicted_label) in enumerate(zip(sample_texts, sample_predictions), start=1):
    print('--- Sample', index)
    print('Predicted venue:', predicted_label)
    print('Text preview:', str(text)[:250])

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best CV accuracy: 0.5722916666666668
Best parameters: {'knn__metric': 'cosine', 'knn__n_neighbors': 7, 'knn__weights': 'distance', 'tfidf__max_features': 10000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Test accuracy: 0.5733333333333334
Confusion matrix:
 [[269  14   3   1   5   0   6   2]
 [ 16 176  19  15  17  19  25  13]
 [  0  14 149   3   8 121   3   2]
 [  1  16   5 127  11   2  12 126]
 [  1  21  22   1 172  31  47   5]
 [  2  20 134   1   8 122  13   0]
 [  4  20   5   9  36   6 215   5]
 [  0  12   4 106   6   4  22 146]]
Classification report:
                                                                       precision    recall  f1-score   support

                                                      Bioinformatics       0.92      0.90      0.91       300
                                   Lecture Notes in Computer Science       0.60      0.59      0.59       300
                                    globa

In [8]:
# Save the trained model and verify it loads correctly.
model_path = 'dblp_knn_model.joblib'
encoder_path = 'dblp_knn_label_encoder.joblib'
joblib.dump(best_model, model_path)
joblib.dump(label_encoder, encoder_path)

loaded_model = joblib.load(model_path)
loaded_encoder = joblib.load(encoder_path)
loaded_predictions = loaded_encoder.inverse_transform(loaded_model.predict(X_test.iloc[:3]))
print('Loaded model predictions:', list(loaded_predictions))

Loaded model predictions: ['global communications conference', 'international conference on robotics and automation', 'international conference on robotics and automation']
